# Portfolio API — Wint Wealth (`slug: wintwealth`)

Exercises all `/portfolio/*` endpoints scoped to the `wintwealth` source.
Covers corporate bonds, NCDs, and Sovereign Gold Bonds (SGBs).

**Auth pre-req:** Log in to [wintwealth.com](https://www.wintwealth.com) inside the AlphaForge Chrome session (`--remote-debugging-port=9299`). Set `WINTWEALTH_USER_ID` in `backend/.env.cred.local`.

In [ ]:
import json, os
from pathlib import Path

SLUG     = "wintwealth"
MODE     = "http"          # "in_process" | "http"
BASE     = "http://localhost:8000/api/v1"
FIXTURES = Path.cwd().parent / "tests" / "fixtures" / "broker_csvs"

# Dev defaults from backend/app/core/config.py — override via env if you've
# changed AlphaForge admin credentials.
AF_USERNAME = os.getenv("AF_USERNAME", "admin")
AF_PASSWORD = os.getenv("AF_PASSWORD", "alphaforge-dev")

if MODE == "in_process":
    from fastapi.testclient import TestClient
    from app.main import app
    client = TestClient(app)
    PREFIX = "/api/v1"
else:
    import httpx
    client = httpx.Client(base_url=BASE, timeout=60.0)
    PREFIX = ""


def _login() -> str:
    r = client.post(
        f"{PREFIX}/auth/token",
        data={"username": AF_USERNAME, "password": AF_PASSWORD},
    )
    if r.status_code != 200:
        raise RuntimeError(
            f"Auth failed ({r.status_code}): {r.text}. "
            "Set AF_USERNAME / AF_PASSWORD env vars if you changed admin creds."
        )
    return r.json()["access_token"]


def _ensure_auth() -> None:
    if "Authorization" not in client.headers:
        client.headers["Authorization"] = f"Bearer {_login()}"


def _request(method: str, path: str, **kw):
    _ensure_auth()
    r = client.request(method, f"{PREFIX}{path}", **kw)
    # Self-heal on token expiry / fresh-kernel state.
    if r.status_code == 401:
        client.headers["Authorization"] = f"Bearer {_login()}"
        r = client.request(method, f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text


def get(path, **kw):  return _request("GET", path, **kw)
def post(path, **kw): return _request("POST", path, **kw)

def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

_ensure_auth()
print(f"Mode: {MODE}  slug: {SLUG}  authed as: {AF_USERNAME}")

## 1. Source info

`status: ready` when `WINTWEALTH_USER_ID` is set, `unconfigured` otherwise.
CSV upload works regardless of status.

In [ ]:
status, body = get(f"/portfolio/sources/{SLUG}")
print(status)
pp(body)

## 2. Sync

Attaches to Chrome via CDP, navigates to `wintwealth.com/portfolio`, reloads
the page, and intercepts the holdings XHR. Result is cached to
`~/.alphaforge/portfolio-dumps/wintwealth-holdings-live.csv`.

> Requires `MODE="http"` with a live server and an open Chrome session
> where you are already logged in to wintwealth.com.

In [ ]:
status, body = post(f"/portfolio/sources/{SLUG}/sync")
print(status, f"  holdings={body.get('holdings_count')}  status={body.get('info', {}).get('status')}")
for h in (body.get("holdings") or [])[:5]:
    asset = h.get('asset_class', '')
    tag   = "[SGB]" if asset == "gold" else "[BOND]"
    print(f"  {tag:6} {str(h.get('name') or h['symbol'])[:40]:40}  qty={h['quantity']:<6}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 3. Upload CSV (offline fallback)

Export from `wintwealth.com` → Portfolio → Download CSV. Drop the file at
`tests/fixtures/broker_csvs/wintwealth_holdings.csv` and re-run this cell.

In [ ]:
csv_path = FIXTURES / "wintwealth_holdings.csv"
if csv_path.exists():
    with csv_path.open("rb") as f:
        r = client.post(
            f"{PREFIX}/portfolio/sources/{SLUG}/upload",
            files={"file": (csv_path.name, f, "text/csv")},
        )
    print(r.status_code)
    body = r.json()
    print(f"Uploaded {body.get('holdings_count')} holdings")
    for h in (body.get("holdings") or [])[:5]:
        print(f"  [{h['asset_class']:11}] {str(h.get('name') or h['symbol'])[:40]:40}  qty={h['quantity']}")
else:
    print(f"No fixture at {csv_path} — drop a Wint Wealth CSV export there.")

## 4. Holdings — wintwealth only

Asset classes will be `bond` (corporate bonds / NCDs) or `gold` (SGBs).

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print(status, "  totals:", body.get("totals"))
print(f"\n{len(body.get('holdings', []))} holdings:")
for h in body.get("holdings", []):
    name = str(h.get('name') or h['symbol'])[:38]
    print(f"  [{h['asset_class']:11}] {name:38}  qty={h['quantity']:<6}  avg=₹{h['avg_price']:>10,.2f}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 5. Allocation (wintwealth)

Expected: majority `bond`, with `gold` if SGBs are held.

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print("Allocation:")
for a in body.get("allocation", []):
    print(f"  {a['asset_class']:12} ₹{a['value']:>14,.0f}  ({a['pct']:>5.1f}%)")

## 6. Treemap (wintwealth)

In [ ]:
status, body = get("/portfolio/treemap", params={"source": SLUG})
print(status)
for c in (body.get("cells") or [])[:10]:
    print(f"  {c['symbol']:20} {c['pct']:>5.1f}% @ ({c['left_pct']:>5.1f}, {c['top_pct']:>5.1f}) {c['width_pct']:>5.1f}x{c['height_pct']:>5.1f}")

## 7. Rebalance (wintwealth)

In [ ]:
status, body = get("/portfolio/rebalance", params={"source": SLUG})
print("Drift:")
for d in body.get("drift", []):
    print(f"  {d['asset_class']:12} target {d['target_pct']:>5.1f}% · actual {d['actual_pct']:>5.1f}% · drift {d['drift_pct']:>+5.1f}%")
print("\nSuggestions:")
for s in body.get("suggestions", []):
    print("  -", s["action"])

## 8. Standalone dump (bypass FastAPI)

Directly runs the CDP fetch + CSV write without starting the server.
Useful for testing auth and CSV output end-to-end.

In [ ]:
import asyncio, sys
sys.path.insert(0, str(Path.cwd().parent))  # add backend/ to path

from app.modules.brokers.wintwealth.wintwealth_dump import dump_wintwealth

path = await dump_wintwealth()
print(f"Dumped → {path}")

## 9. Reset wintwealth cache

In [ ]:
if MODE == "in_process":
    from app.modules.brokers.registry import SOURCES
    SOURCES[SLUG].reset()
    status, body = get(f"/portfolio/sources/{SLUG}")
    print(f"{SLUG}: status={body['status']}  holdings={body['holdings_count']}")
else:
    print("Switch MODE to 'in_process' to reset the in-memory cache directly.")
    print("Or restart the server to clear all cached holdings.")